# Advanced Chemical Reactor Engineering - Group: Stirred not Shaken
### Spinning Disc Reactor (SDR) Simulation: Glucose → Fructose → HMF → FDCA

This notebook simulates the **Spinning Disc Reactor (SDR)** for the catalytic oxidation of glucose to FDCA.

### Reaction pathway
$$\text{Glucose} \xrightarrow{k_1} \text{Fructose} \xrightarrow{k_3} \text{HMF} \xrightarrow{k_6} \text{FDCA}$$

Side reactions from HMF:
- $\text{HMF} \xrightarrow{k_4} \text{Humins}$
- $\text{HMF} \xrightarrow{k_5} \text{Levulinic acid + Formic acid}$

### SDR modelling choice used here
In the report, the SDR is described as a **reactor with a narrow RTD** and approximated as **$N$ CSTRs in series** for direct comparison with the CSTR. That is the approach used in this notebook.

- **Local SDR hydrodynamics** are used to estimate the high mass-transfer coefficients ($k_La$)
- **The reactor model** is then represented as **$N$ ideal mixed stages in series**
- **The total reactor holdup and mean residence time** are kept equal to the CSTR basis so the comparison focuses on **RTD + mass transfer**, not on using a completely different reactor size

This gives a practical process-model approximation of an SDR train / numbering-up concept while keeping the mole balances identical to the CSTR model.

### Reference
Meeuwse, M. (2011). *Rotor-stator spinning disc reactor*. PhD Thesis, Technische Universiteit Eindhoven. DOI: 10.6100/IR702643

### 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings('ignore')

### 1. Physical Properties

All properties are evaluated at the operating temperature **T = 160 °C (433.15 K)**.

#### Diffusivities — Wilke–Chang correction
$$D(T, \mu) = D_{\mathrm{ref}} \cdot \frac{T}{T_{\mathrm{ref}}} \cdot \frac{\mu_{\mathrm{ref}}}{\mu}$$

This captures the increase in diffusion with temperature and the decrease in diffusion resistance as viscosity drops.

#### SDR-specific film thickness
For the spinning disc, the local film thickness is estimated with:
$$\delta = \left(\frac{3\nu Q_L}{2\pi^2 \Omega^2 R_d^2}\right)^{1/3}$$
where $\nu$ is the kinematic viscosity, $Q_L$ is the liquid flow per disc, $\Omega$ is the angular velocity, and $R_d$ is the disc radius.

In [ ]:
T, g, R = 160 + 273.15, 9.81, 8.3145   # K, m/s^2, J/mol/K
rho     = 780.0     # kg/m^3  density MIBK
mu      = 0.35e-3   # Pa.s    dynamic viscosity MIBK
nu      = mu / rho  # m^2/s   kinematic viscosity
sigma   = 12e-3     # N/m     interfacial tension MIBK/water

# Diffusivities corrected to operating T and mu via Wilke-Chang
mu_ref = 0.585e-3
D_HMF  = 6.0e-10 * (T/298) * (mu_ref/mu)   # m^2/s  HMF in MIBK
D_O2   = 3.5e-9  * (T/298) * (mu_ref/mu)   # m^2/s  O2 in MIBK

print(f"T = {T:.2f} K")
print(f"D_HMF = {D_HMF:.3e} m^2/s")
print(f"D_O2  = {D_O2:.3e} m^2/s")

### 2. SDR Geometry and Modelling Basis

This notebook separates two things:

1. **Single-disc hydrodynamics** used to estimate SDR mass transfer coefficients
2. **Process-scale reactor model** used for comparison with the CSTR

The local disc geometry is kept at lab / pilot scale to estimate thin-film hydrodynamics, while the **overall reactor model** is put on the **same 100 m³, 1 h basis as the CSTR notebook**. That way, differences in performance mainly come from:

- higher $k_La$ values in the SDR
- narrower residence time distribution, approximated by **$N$ CSTRs in series**

So the local disc volume is **not** used directly as the total process holdup.

In [ ]:
# --- Local single-disc geometry used for SDR hydrodynamics ---
R_d   = 0.135      # m      disc radius
h_gap = 1.0e-3     # m      rotor-stator gap
n_rot = 50.0       # rev/s  rotation speed
Omega = 2 * np.pi * n_rot   # rad/s

# Local liquid flow handled by one disc (used only for hydrodynamic estimates)
Q_L_disc   = 5.0e-5         # m^3/s
Q_aq_disc  = Q_L_disc / 3.0
Q_org_disc = Q_L_disc * 2.0 / 3.0

# Thin-film thickness on one spinning disc
delta = (3 * nu * Q_L_disc / (2 * np.pi**2 * Omega**2 * R_d**2))**(1/3)

A_disc   = np.pi * R_d**2
V_film   = A_disc * delta
V_gap    = A_disc * h_gap
V_single_disc = V_film + V_gap

tau_single_disc = V_single_disc / Q_L_disc

# --- Process-scale comparison basis (same as CSTR notebook) ---
V_total_model = 100.0      # m^3 total reactive holdup for fair comparison
N_stages      = 10         # SDR approximated as N CSTRs in series

eps_g_model   = 0.20
eps_s_model   = 0.10
eps_l_model   = 1.0 - eps_g_model - eps_s_model

eps_aq_liq    = 1/3
eps_org_liq   = 1 - eps_aq_liq

V_g_total   = eps_g_model * V_total_model
V_p_total   = eps_s_model * V_total_model
V_liq_total = eps_l_model * V_total_model
V_aq_total  = V_liq_total * eps_aq_liq
V_org_total = V_liq_total * eps_org_liq

print(f"Single-disc radius         R_d = {R_d*100:.1f} cm")
print(f"Single-disc speed            = {n_rot:.1f} rev/s ({n_rot*60:.0f} RPM)")
print(f"Film thickness           delta = {delta*1e6:.1f} µm")
print(f"Single-disc liquid volume     = {V_single_disc*1e6:.2f} mL")
print(f"Single-disc residence time    = {tau_single_disc:.2f} s")
print()
print("Comparison basis for reactor model:")
print(f"  Total reactor holdup  = {V_total_model:.1f} m^3")
print(f"  Number of SDR stages  = {N_stages:d}")
print(f"  Total phase volumes   = V_aq={V_aq_total:.2f}  V_org={V_org_total:.2f}  V_p={V_p_total:.2f} m^3")

### 3. SDR Hydrodynamics and Power Input

#### Local rotor power and energy dissipation
The local mass-transfer coefficients are estimated from the rotor–stator hydrodynamics of a single spinning disc:
$$P = C_M \cdot \frac{1}{2}\rho\Omega^3R_d^5$$
with
$$C_M = \frac{3.87}{Re_\Omega^{1/2}}, \qquad Re_\Omega = \frac{\Omega R_d^2}{\nu}$$

This gives the very high local energy dissipation responsible for the enhanced gas–liquid and liquid–liquid transport.

In [ ]:
v_SG  = 0.05   # m/s  superficial gas velocity in rotor-stator gap

# Local hydrodynamics on one disc
Re_Omega = Omega * R_d**2 / nu
C_M      = 3.87 / Re_Omega**0.5
P_disc   = C_M * 0.5 * rho * Omega**3 * R_d**5
PV_local = P_disc / V_single_disc

eps_diss = PV_local / rho      # m^2/s^3 specific energy dissipation
Re_G     = v_SG * h_gap / nu
Sc_O2    = nu / D_O2
Sc_HMF   = nu / D_HMF

# Meeuwse-style gas-liquid kLa correlation for rotor-stator SDR
C1_SDR, alpha_SDR, beta_SDR = 0.18, 0.75, 0.25
kLa_GL = C1_SDR * Re_Omega**alpha_SDR * Re_G**beta_SDR * Sc_O2**0.5 * D_O2 / h_gap**2

# Gas holdup and bubble size in the thin gap
eps_g_local = min(0.15 * (Re_Omega / 1e5)**0.3 * (v_SG / 0.01)**0.5, 0.25)
d_b = 4.15 * (sigma / rho)**0.6 * (P_disc / V_single_disc)**(-0.4) * v_SG**0.5 + 1e-4
d_b = max(d_b, 1e-4)

print(f"Re_Omega     = {Re_Omega:.2e}")
print(f"Re_G         = {Re_G:.2e}")
print(f"P_disc       = {P_disc:.2f} W")
print(f"P/V (local)  = {PV_local:.2e} W/m^3")
print(f"eps_diss     = {eps_diss:.4f} m^2/s^3")
print(f"kLa_GL       = {kLa_GL:.4f} s^-1")
print(f"eps_g_local  = {eps_g_local:.3f}")
print(f"d_b          = {d_b*1e3:.2f} mm")

### 4. Mass Transfer Coefficients

Three interfacial mass transfer steps are modelled, each with a volumetric coefficient $k_La$ ($s^{-1}$).

#### A1 — Gas → Organic liquid (O₂ absorption) — SDR gap
From the Meeuwse (2011) correlation applied in the previous cell.

Interfacial area per unit volume in the gap:
$$a_{GL} = \frac{6\,\varepsilon_g}{d_b}$$

#### A2 — Aqueous → Organic liquid (HMF extraction) — film surface
The thin film geometry on the disc provides a well-defined liquid–liquid interface.  
The film-renewal model gives:
$$k_{LL} = 2\sqrt{\frac{D_{HMF} \Omega}{\pi}}$$

Interfacial area per unit volume (disc film surface / film volume):
$$a_{LL} = \frac{A_{\text{disc}}}{V_{\text{film}}} = \frac{1}{\delta}$$

#### A3 — Organic liquid → Catalyst particle (HMF and $O_2$) — film interior
Same Armenante & Kirwan (1989) correlation as the CSTR, using the SDR specific energy dissipation $\varepsilon$:
$$a_{LS} = \frac{6\,\varepsilon_s}{d_p}$$

In [ ]:
# A1: Gas-liquid (O2: gas -> organic) from local SDR hydrodynamics
# kLa_GL was already computed in the previous cell

# A2: Liquid-liquid (HMF: aq -> organic) in thin film
k_LL   = 2 * np.sqrt(D_HMF * Omega / np.pi)   # m/s
a_LL   = 1.0 / delta                           # m^2/m^3 thin-film area density
kLa_LL = k_LL * a_LL                           # s^-1

# A3: Liquid-solid (HMF & O2: organic -> catalyst)
d_p = 5e-6               # m catalyst particle diameter
a_LS  = 6 * eps_s_model / d_p
Re_LS = (eps_diss * d_p**4 / nu**3)**0.25

Sh_HMF     = 2 + 0.36 * Re_LS**0.75 * (nu / D_HMF)**0.33
kLa_LS_HMF = Sh_HMF * D_HMF / d_p * a_LS

Sh_O2      = 2 + 0.36 * Re_LS**0.75 * (nu / D_O2)**0.33
kLa_LS_O2  = Sh_O2  * D_O2  / d_p * a_LS

print(f"kLa_GL     (O2  gas->org) = {kLa_GL:.4f} s^-1")
print(f"kLa_LL     (HMF aq->org)  = {kLa_LL:.4f} s^-1")
print(f"kLa_LS_HMF (HMF org->cat) = {kLa_LS_HMF:.2f} s^-1")
print(f"kLa_LS_O2  (O2  org->cat) = {kLa_LS_O2:.2f} s^-1")

### 5. Reaction Kinetics

Identical to the CSTR model — kinetics are reactor-independent.

#### Isomerisation and dehydration (aqueous phase)
All rate constants are first-order ($s^{-1}$):

| Reaction | Symbol | Value (s⁻¹) |
|----------|--------|-------------|
| Glucose $\rightarrow$ Fructose | $k_1$ | 0.104/60 |
| Fructose $\rightarrow$ Glucose (reverse) | $k_2$ | 0.052/60 |
| Fructose $\rightarrow$ HMF | $k_3$ | 0.286/60 |
| HMF $\rightarrow$ Humins | $k_4$ | 0.013/60 |
| HMF $\rightarrow$ LA + FA | $k_5$ | 0.031/60 |

#### Oxidation to FDCA (catalyst particle surface)
Two parallel series paths via DFF and HFCA:
$$k_6 = \frac{1}{\frac{1}{k_{\text{HMF→DFF}}} + \frac{1}{k_{\text{DFF→FFCA}}}} + \frac{1}{\frac{1}{k_{\text{HMF→HFCA}}} + \frac{1}{k_{\text{HFCA→FFCA}}}}$$

#### Thiele modulus (SDR film geometry)
For 5-µm catalyst particles the Thiele modulus and internal effectiveness factor remain the same as the CSTR:
$$\phi = \frac{d_p}{6}\sqrt{\frac{k_6}{D_{\text{HMF}}}}, \quad \eta = \frac{1}{\phi}\left(\frac{1}{\tanh(3\phi)} - \frac{1}{3\phi}\right)$$

In [ ]:
# Aqueous-phase kinetics (same as CSTR notebook)
k1, k2 = 0.104/60, 0.052/60    # s^-1  Glu <-> Fru
k3      = 0.286/60             # s^-1  Fru -> HMF
k4      = 0.013/60             # s^-1  HMF -> Humins
k5      = 0.031/60             # s^-1  HMF -> LA + FA

# Oxidation kinetics on catalyst surface (same basis as CSTR notebook)
k_HMF_DFF   = 0.0693           # m^3/mol/s
k_DFF_FFCA  = 0.0273           # m^3/mol/s
k_HMF_HFCA  = 0.0319           # m^3/mol/s
k_HFCA_FFCA = 2.07e-3          # m^3/mol/s

k6 = 1/(1/k_HMF_DFF + 1/k_DFF_FFCA) + 1/(1/k_HMF_HFCA + 1/k_HFCA_FFCA)

# Internal diffusion check
phi = (d_p / 6) * np.sqrt(k6 / D_HMF)
eta = (1/np.tanh(3*phi) - 1/(3*phi)) / phi

print(f"k6  (oxidation, combined) = {k6:.5f} m^3/mol/s")
print(f"Thiele modulus  phi       = {phi:.5f}")
print(f"Effectiveness   eta       = {eta:.6f}")

### 6. O₂ Saturation Concentration — Henry's Law

The operating pressure in the report is **20 bar total**. The oxygen partial pressure is therefore:
$$P_{O_2} = P_{\mathrm{total}} - P_{\mathrm{vap,water}} - P_{\mathrm{vap,MIBK}}$$

The dissolved oxygen saturation concentration in the organic phase is then found from Henry's law:
$$H(T) = H_{298}\,\exp\!\left[\frac{\Delta H_{\mathrm{sol}}}{R}\left(\frac{1}{298} - \frac{1}{T}\right)\right], \qquad C^*_{O_2} = \frac{P_{O_2}}{H(T)}$$

In [ ]:
P_total    = 20.0e5                                        # Pa total operating pressure (20 bar)
P_aq_vap   = 10**(3.55959 - 643.748/(T - 198.043)) * 1e5   # Pa water vapour pressure
P_org_vap  = 10**(3.95298 - 1254.095/(T - 71.537)) * 1e5   # Pa MIBK vapour pressure
P_O2       = P_total - P_aq_vap - P_org_vap                # Pa oxygen partial pressure

if P_O2 <= 0:
    raise ValueError("No oxygen partial pressure left after subtracting vapour pressures.")

H_O2_298 = 101.3e3 / (8.71e-4 * (780e3/58.08))
H_O2     = H_O2_298 * np.exp(15e3 / R * (1/298 - 1/T))
C_O2_sat = P_O2 / H_O2

print(f"P_total      = {P_total/1e5:.2f} bar")
print(f"P_vap water  = {P_aq_vap/1e5:.2f} bar")
print(f"P_vap MIBK   = {P_org_vap/1e5:.2f} bar")
print(f"P_O2         = {P_O2/1e5:.2f} bar")
print(f"H_O2         = {H_O2:.2e} Pa·m^3/mol")
print(f"C*_O2        = {C_O2_sat:.4f} mol/m^3")

### 7. SDR Operating Parameters — $N$ CSTRs in Series Approximation

For the report, the SDR is approximated as **$N$ CSTRs in series**. This keeps the same mole balances as the CSTR while representing the narrower RTD of the SDR.

#### Modelling basis used here
- **Total reactor holdup:** same as CSTR notebook ($100\,\mathrm{m^3}$)
- **Total mean residence time:** same as CSTR notebook ($\tau_{\mathrm{total}} = 1\,\mathrm{h}$)
- **RTD approximation:** $N$ equal CSTR stages in series
- **Mass transfer coefficients:** taken from the SDR thin-film / rotor–stator hydrodynamics

This means the notebook is set up for a **fair conceptual comparison** with the CSTR rather than as a literal single-disc equipment sizing model.

In [ ]:
C_Glu_feed = 1500.0
m_AO       = 0.77

tau_total = 3600.0                   # s total mean residence time (same as CSTR)
tau_stage = tau_total / N_stages

F_aq  = V_aq_total  / tau_total      # m^3/s
F_org = V_org_total / tau_total      # m^3/s

V_aq_stage  = V_aq_total  / N_stages
V_org_stage = V_org_total / N_stages
V_p_stage   = V_p_total   / N_stages

Q_total_process  = F_aq + F_org
n_parallel_discs = Q_total_process / Q_L_disc

print(f"Total SDR-train mean residence time = {tau_total/3600:.2f} h")
print(f"Residence time per stage            = {tau_stage:.1f} s")
print(f"Number of stages                    = {N_stages:d}")
print(f"Process liquid flow                 = {Q_total_process:.5f} m^3/s")
print(f"Local flow handled per disc         = {Q_L_disc:.5f} m^3/s")
print(f"Parallel discs needed at this local hydraulic loading ≈ {n_parallel_discs:.0f}")

### 8. ODE System — One SDR Stage

Each SDR stage uses the **same multiphase mole balances as the CSTR**, but with:
- the **SDR mass-transfer coefficients**
- a **smaller stage volume** ($V/N$)
- an **inlet from the previous stage**

So for each stage:
$$\frac{dC_i}{dt} = \frac{F}{V_{\mathrm{stage}}}(C_{i,\mathrm{in}} - C_i) + \sum r_j \pm J_{\mathrm{MT}}$$

The catalyst-phase states still have **no convective term**, exactly as in the CSTR notebook, because they represent local particle-surface concentrations inside each stage.

In [ ]:
def sdr_stage_odes(t, y, y_in):
    Glu, Fru, HMF_aq, HMF_org, HMF_p, O2_org, O2_p, Hum, LA = [max(v, 0.0) for v in y]
    Glu_in, Fru_in, HMF_aq_in, HMF_org_in, _, O2_org_in, _, Hum_in, LA_in = y_in

    # Reaction rates [mol/m^3_phase/s]
    r1 = k1 * Glu
    r2 = k2 * Fru
    r3 = k3 * Fru
    r4 = k4 * HMF_aq
    r5 = k5 * HMF_aq
    r6 = k6 * eta * HMF_p * O2_p

    # Mass transfer fluxes [mol/m^3_source/s]
    J_GL  = kLa_GL * (C_O2_sat - O2_org)
    J_LL  = kLa_LL * (HMF_aq - m_AO * HMF_org)
    J_HMF = kLa_LS_HMF * (HMF_org - HMF_p)
    J_O2  = kLa_LS_O2  * (O2_org  - O2_p)

    # CSTR-stage balances (same structure as the CSTR notebook)
    dGlu    = F_aq  / V_aq_stage  * (Glu_in     - Glu)     - r1 + r2
    dFru    = F_aq  / V_aq_stage  * (Fru_in     - Fru)     + r1 - r2 - r3
    dHMFaq  = F_aq  / V_aq_stage  * (HMF_aq_in  - HMF_aq)  + r3 - r4 - r5 - J_LL
    dHMForg = F_org / V_org_stage * (HMF_org_in - HMF_org) + J_LL * (V_aq_stage / V_org_stage) - J_HMF
    dO2org  = F_org / V_org_stage * (O2_org_in  - O2_org)  + J_GL - J_O2

    # Catalyst states remain local to the stage (no convective term)
    dHMFp = J_HMF * (V_org_stage / V_p_stage) - r6
    dO2p  = J_O2  * (V_org_stage / V_p_stage) - r6

    dHum = F_aq / V_aq_stage * (Hum_in - Hum) + r4
    dLA  = F_aq / V_aq_stage * (LA_in  - LA)  + r5

    return [dGlu, dFru, dHMFaq, dHMForg, dHMFp, dO2org, dO2p, dHum, dLA]

### 9. Solving the SDR Train

The SDR is solved **stage-by-stage** at steady state:

1. Solve stage 1 using the fresh feed as inlet
2. Use the outlet of stage 1 as the inlet to stage 2
3. Repeat until stage $N$

The final stage outlet is the overall SDR outlet.

This gives a practical approximation of a reactor with a narrow RTD without changing the chemistry or mass-balance structure.

In [ ]:
# Fresh feed to stage 1
# Organic feed is taken as initially free of HMF and dissolved oxygen;
# oxygen is supplied through interphase mass transfer inside each stage.
y_feed = np.array([C_Glu_feed, 0.0, 0.0, 0.0, 0.0, 0.0, C_O2_sat, 0.0, 0.0], dtype=float)

stage_outlets = []
stage_rates_r6 = []
stage_success = []

for j in range(N_stages):
    y_in = y_feed.copy() if j == 0 else stage_outlets[-1].copy()

    # Numerical initial guess for the local stage states
    y0_stage = y_in.copy()
    y0_stage[4] = max(y_in[3], 0.0)   # HMF on catalyst starts near org concentration
    y0_stage[6] = C_O2_sat            # oxygen on catalyst starts near saturation

    sol_stage = solve_ivp(
        lambda t, y: sdr_stage_odes(t, y, y_in),
        [0, 8 * tau_stage],
        y0_stage,
        method='BDF',
        rtol=1e-9,
        atol=1e-11,
        dense_output=False,
    )

    y_out = np.clip(sol_stage.y[:, -1], 0.0, None)
    stage_outlets.append(y_out)
    stage_rates_r6.append(k6 * eta * y_out[4] * y_out[6])
    stage_success.append(sol_stage.success)

stage_outlets = np.array(stage_outlets)
stage_rates_r6 = np.array(stage_rates_r6)

ss = stage_outlets[-1]
Glu_ss, Fru_ss, HMFaq_ss, HMForg_ss, HMFp_ss, O2org_ss, O2p_ss, Hum_ss, LA_ss = ss

print(f"All stage integrations successful: {all(stage_success)}")
print("Overall SDR outlet concentrations [mol/m^3]:")
print(f"  Glu    = {Glu_ss:.3f}")
print(f"  Fru    = {Fru_ss:.3f}")
print(f"  HMF_aq = {HMFaq_ss:.6f}")
print(f"  HMF_org= {HMForg_ss:.6f}")
print(f"  HMF_p  = {HMFp_ss:.6f}")
print(f"  O2_org = {O2org_ss:.6f}")
print(f"  O2_p   = {O2p_ss:.6f}")
print(f"  Humins = {Hum_ss:.6f}")
print(f"  LA=FA  = {LA_ss:.6f}")

### 10. Outlet Performance Metrics

#### Glucose conversion
$$X_{\mathrm{Glu}} = \frac{C_{\mathrm{Glu,feed}} - C_{\mathrm{Glu,out}}}{C_{\mathrm{Glu,feed}}}$$

#### Selectivity
For convected byproducts we use the **outlet molar flow**. For FDCA, which is not an explicit state variable, the total production rate is calculated from the **sum of the reaction rate over all SDR stages**:
$$F_{\mathrm{FDCA}} = \sum_{j=1}^{N} r_{6,j}\,V_{p,\mathrm{stage}}$$

In [ ]:
X_Glu = (C_Glu_feed - Glu_ss) / C_Glu_feed

# Total FDCA production over the whole SDR train
F_FDCA = np.sum(stage_rates_r6 * V_p_stage)      # mol/s

# Convected byproducts from overall outlet
F_Hum = F_aq * Hum_ss                            # mol/s
F_LA  = F_aq * LA_ss                             # mol/s
F_FA  = F_LA                                     # mol/s

F_Glu_rxd = F_aq * (C_Glu_feed - Glu_ss)

if F_Glu_rxd > 1e-15:
    S_FDCA = F_FDCA / F_Glu_rxd
    S_Hum  = F_Hum  / F_Glu_rxd
    S_LA   = F_LA   / F_Glu_rxd
else:
    S_FDCA = S_Hum = S_LA = 0.0

MW_FDCA, MW_Hum, MW_LA, MW_FA = 168.11, 126.11, 116.12, 46.03

print("="*60)
print("SDR TRAIN OUTLET RESULTS")
print("="*60)
print(f"Glucose conversion   X  = {X_Glu*100:.2f}%")
print(f"FDCA selectivity     S  = {S_FDCA*100:.2f}%")
print(f"Humins selectivity      = {S_Hum*100:.3f}%")
print(f"LA+FA selectivity       = {S_LA*100:.3f}%")
print()
print(f"FDCA   = {F_FDCA*MW_FDCA/1000*3600:.2f} kg/h")
print(f"Humins = {F_Hum*MW_Hum/1000*3600:.4f} kg/h")
print(f"LA     = {F_LA*MW_LA/1000*3600:.4f} kg/h")
print(f"FA     = {F_FA*MW_FA/1000*3600:.5f} kg/h")
print()
print(f"Total residence time = {tau_total/3600:.2f} h across {N_stages:d} SDR stages")

### 11. Rate-Determining Step (RDS) Analysis

The RDS analysis is reported on the **same total reactor-volume basis as the CSTR**, so the rates can be compared directly.

The step with the smallest maximum possible rate is the bottleneck.

In [ ]:
K_eq = k1 / k2
C_Fru_max     = K_eq / (1 + K_eq) * C_Glu_feed
C_HMF_aq_max  = C_Glu_feed
C_HMF_org_max = m_AO * C_HMF_aq_max

rds = {
    'k1  Glu->Fru  (aq)':  k1 * C_Glu_feed * V_aq_total,
    'k3  Fru->HMF  (aq)':  k3 * C_Fru_max * V_aq_total,
    'LL  HMF aq->org':     kLa_LL * C_HMF_aq_max * V_aq_total,
    'LS  HMF org->cat':    kLa_LS_HMF * C_HMF_org_max * V_org_total,
    'GL  O2  gas->org':    kLa_GL * C_O2_sat * V_org_total,
    'LS  O2  org->cat':    kLa_LS_O2 * C_O2_sat * V_org_total,
    'k6*eta  reaction':    k6 * eta * C_HMF_org_max * C_O2_sat * V_p_total,
}

print("="*60)
print("SDR RDS -- MAXIMUM DRIVING FORCE [mol/s]")
print("="*60)
min_rate = min(rds.values())
for name, rate in rds.items():
    tag = "  <-- BOTTLENECK" if rate == min_rate else ""
    print(f"  {name:25s}  {rate:.3e}{tag}")

### 12. Results — Stagewise Profiles Through the SDR Train

Because the SDR is represented as **$N$ ideal stages in series**, the results below are plotted as **stagewise outlet values** against cumulative mean residence time.

So the x-axis is not the startup transient of a stirred vessel. It represents progress through the SDR stage train from inlet to outlet.

In [ ]:
# Stagewise profiles
x_norm = np.arange(0, N_stages + 1) / N_stages

# Include feed as stage 0 for plotting
profiles = np.vstack([y_feed, stage_outlets])

# Cumulative product rates through the train
FDCA_stage_kgph = stage_rates_r6 * V_p_stage * 3600 * MW_FDCA / 1000
FDCA_cum_kgph   = np.concatenate([[0.0], np.cumsum(FDCA_stage_kgph)])
FDCA_cum_molps  = np.concatenate([[0.0], np.cumsum(stage_rates_r6 * V_p_stage)])
Hum_out = F_aq * profiles[:, 7] * 3600 * MW_Hum / 1000
LA_out  = F_aq * profiles[:, 8] * 3600 * MW_LA  / 1000
FA_out  = F_aq * profiles[:, 8] * 3600 * MW_FA  / 1000

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(
    'SDR approximation: N CSTRs in series for Glucose -> Fructose -> HMF -> FDCA\n'
    f'T = 160 °C,  P = 20 bar total,  tau_total = {tau_total/3600:.1f} h,  N = {N_stages:d}',
    fontsize=11, fontweight='bold')
plt.subplots_adjust(hspace=0.40, wspace=0.35, left=0.07, right=0.97, top=0.88, bottom=0.09)

xlabel_lbl = 'Normalised position through SDR train [-]'

# Panel 1 -- Sugars
ax = axes[0, 0]
ax.plot(x_norm, profiles[:, 0], marker='o', label='Glucose')
ax.plot(x_norm, profiles[:, 1], marker='o', label='Fructose')
ax.set(xlabel=xlabel_lbl, ylabel='Concentration (mol/m³)', title='Sugars (aq)')
ax.legend(); ax.grid(alpha=0.3)

# Panel 2 -- HMF across phases
ax = axes[0, 1]
ax.plot(x_norm, profiles[:, 2], marker='o', label='HMF (aq)')
ax.plot(x_norm, profiles[:, 3], marker='o', label='HMF (org)')
ax.plot(x_norm, profiles[:, 4], marker='o', ls='--', label='HMF (particle)')
ax.set(xlabel=xlabel_lbl, ylabel='Concentration (mol/m³)', title='HMF — All Phases')
ax.legend(); ax.grid(alpha=0.3)

# Panel 3 -- Byproducts
ax = axes[0, 2]
ax.plot(x_norm, profiles[:, 7], marker='o', label='Humins (aq)')
ax.plot(x_norm, profiles[:, 8], marker='o', ls='--', label='LA = FA (aq)')
ax.set(xlabel=xlabel_lbl, ylabel='Concentration (mol/m³)', title='Byproducts (aq)')
ax.legend(); ax.grid(alpha=0.3)

# Panel 4 -- O2
ax = axes[1, 0]
ax.plot(x_norm, profiles[:, 5], marker='o', label='O2 (org)')
ax.plot(x_norm, profiles[:, 6], marker='o', ls='--', label='O2 (particle)')
ax.axhline(C_O2_sat, color='steelblue', ls=':', lw=1, label=f'C* = {C_O2_sat:.2f}')
ax.set(xlabel=xlabel_lbl, ylabel='Concentration (mol/m³)', title='O2')
ax.legend(); ax.grid(alpha=0.3)

# Panel 5 -- Performance
ax  = axes[1, 1]
ax2 = ax.twinx()
X_profile = (C_Glu_feed - profiles[:, 0]) / C_Glu_feed * 100
F_Glu_profile = F_aq * (C_Glu_feed - profiles[:, 0])
with np.errstate(divide='ignore', invalid='ignore'):
    S_profile = np.where(F_Glu_profile > 1e-15, FDCA_cum_molps / F_Glu_profile * 100, 0)

l1, = ax.plot(x_norm, X_profile, marker='o', label='Glu conversion (%)')
l2, = ax2.plot(x_norm, S_profile, marker='o', ls='--', label='FDCA selectivity (%)')
ax.set(xlabel=xlabel_lbl, ylabel='Conversion (%)', ylim=(0, 105), title='Performance')
ax2.set_ylabel('Selectivity (%)')
ax2.set_ylim(0, 105)
ax.legend([l1, l2], [l.get_label() for l in [l1, l2]], loc='center right')
ax.grid(alpha=0.3)

# Panel 6 -- Product flow rates
ax = axes[1, 2]
ax.plot(x_norm, FDCA_cum_kgph, marker='o', label=f'FDCA cumulative ({F_FDCA*MW_FDCA/1000*3600:.1f} kg/h)')
ax.plot(x_norm, Hum_out,       marker='o', label=f'Humins out ({F_Hum*MW_Hum/1000*3600:.4f} kg/h)')
ax.plot(x_norm, LA_out,        marker='o', ls='--', label=f'LA out ({F_LA*MW_LA/1000*3600:.4f} kg/h)')
ax.plot(x_norm, FA_out,        marker='o', ls='-.', label=f'FA out ({F_FA*MW_FA/1000*3600:.5f} kg/h)')
ax.set(xlabel=xlabel_lbl, ylabel='Product rate (kg/h)', title='Product Rates')
ax.legend(fontsize=8.3)
ax.grid(alpha=0.3)

plt.tight_layout()
#plt.savefig('/mnt/data/SDR_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved to /mnt/data/SDR_results.png')

### 13. CSTR vs. SDR — Key Differences Used in This Notebook

| Property | CSTR notebook | SDR notebook |
|----------|---------------|--------------|
| Flow model | Single perfectly mixed tank | $N$ CSTRs in series |
| RTD | Broad | Narrower |
| Mean residence time | 1 h | 1 h (same comparison basis) |
| Total reactor holdup | 100 m³ | 100 m³ (same comparison basis) |
| Mass transfer | CSTR correlations | SDR thin-film / rotor–stator correlations |
| Main SDR benefit represented here | — | Higher $k_La$ and less back-mixing |

This notebook is therefore suitable for the **concept comparison in the report**: same chemistry, same holdup basis, but different hydrodynamics and RTD.  
A literal single-disc equipment design would require a separate scale-up / numbering-up model.